# Qwen3-4B text-to-SQL — run 4

Beats run 1 or it does not ship. Run 1 scored **84.0%** execution accuracy on a
held-out set of 300; anything here is measured against that same set.

What changed since run 3:

- **Three data sources**: `gretelai/synthetic_text_to_sql`, `b-mc2/sql-create-context`
  (Spider/WikiSQL derived), and synthetic multi-level aggregation.
- **The multi-level generator is diverse now.** Run 3's version emitted one query
  shape per kind with fixed column aliases; the model scored 100% on a held-out
  set from that same generator and 1/6 on hand-written questions — it had
  memorised a template. Aliases, phrasings, schema shapes and SQL formulations
  are all randomised now.
- **Evaluation is hand-written.** 18 multi-level questions the generator never
  produced, so the score measures transfer rather than recall.

**Sidebar settings:** Accelerator `GPU T4 x2`, Internet **On**. No Hugging Face
token needed — outputs are downloaded from this notebook.

Source: https://github.com/garvbahl37-gif/text2sql-qwen3-finetune.git

In [ ]:
# --- 1. Hardware check (stops here if the GPU is unusable) ------------------
import subprocess, torch

name = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
major, minor = torch.cuda.get_device_capability()
print(f"GPU        : {name}")
print(f"capability : {major}.{minor}")
print(f"torch      : {torch.__version__}   bf16: {torch.cuda.is_bf16_supported()}")

if major < 7:
    raise SystemExit(
        f"\nSTOP. Compute capability {major}.{minor} ({name.split(',')[0]}) has no kernels "
        "in modern PyTorch builds.\nFIX: sidebar -> Accelerator -> 'GPU T4 x2', then Run All. "
        "The GPU type cannot be set through the Kaggle API."
    )
print("\nGPU supported." + ("" if torch.cuda.is_bf16_supported() else " Turing has no bf16; fp16 is selected automatically."))

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub

In [ ]:
# --- 2. Verify the install before spending GPU time on it -------------------
try:
    from unsloth import FastLanguageModel, is_bfloat16_supported
    import trl, peft, transformers
    print(f"ok | transformers {transformers.__version__} | trl {trl.__version__} | peft {peft.__version__}")
except Exception as e:
    print("INSTALL FAILED:", type(e).__name__, e)
    print("\nFallback, then Run > Restart session and skip the install cell:")
    print("  !pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo")

In [ ]:
# --- 3. Get the code -------------------------------------------------------
import os, shutil, subprocess
from pathlib import Path

WORK = Path("/kaggle/working/ft")
if not (WORK / "train.py").exists():
    shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
    subprocess.run(["git","clone","--depth","1","https://github.com/garvbahl37-gif/text2sql-qwen3-finetune.git","/kaggle/working/repo"], check=True)
    shutil.copytree("/kaggle/working/repo/training", WORK, dirs_exist_ok=True)
os.chdir(WORK)
print("cwd:", os.getcwd())

# A failing `!python x.py` returns non-zero but does not raise in Jupyter, so the
# notebook would sail past a dead step and fail later somewhere confusing.
def step(cmd: str):
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.run(cmd, shell=True)
    if p.returncode != 0:
        raise SystemExit(f"\nStep failed (exit {p.returncode}):\n  {cmd}")
    print("\nok\n", flush=True)

## 4. Build the training set

12,000 from the two real datasets plus 4,000 synthetic multi-level examples, so
multi-level is 25% of the mix. Every example is executed and checked before it
is kept.

In [ ]:
step("python prepare_data.py --sources gretel,createcontext "
     "--train-size 12000 --val-size 400 --test-size 300")
step("python gen_multilevel.py --n 4000 --out data/multilevel.jsonl")
step("python make_holdout.py --out data/multilevel_holdout.jsonl")

In [ ]:
# --- combine, then refuse to train on anything that would be truncated ------
import json, random
base  = [json.loads(l) for l in open("data/train.jsonl")]
multi = [json.loads(l) for l in open("data/multilevel.jsonl")]
combined = base + multi
random.Random(42).shuffle(combined)
with open("data/train.jsonl","w") as f:
    for r in combined: f.write(json.dumps(r)+"\n")
print(f"train set: {len(combined)} = {len(base)} real + {len(multi)} multi-level "
      f"({len(multi)/len(combined):.0%})")

MAX_SEQ = 768   # the diverse generator reaches ~720 tokens; 640 truncated 5% of it
step(f"python check_lengths.py --data data/train.jsonl --max-seq {MAX_SEQ}")

## 5. Train

Watch `it/s` in the first 50 steps and multiply out before walking away. If it
reports out of memory, halve `--batch-size` and double `--grad-accum`.

In [ ]:
step(f"python train.py --data data --out outputs/run4 "
     f"--max-seq {MAX_SEQ} --batch-size 8 --grad-accum 4 --rank 32 --epochs 1")

## 6. Evaluate

Two sets. The general one is what run 1 scored 84.0% on, so it says whether this
is actually better. The hand-written one says whether multi-level transferred,
rather than whether a template was memorised.

In [ ]:
step("python evaluate.py --adapter outputs/run4 --test data/test.jsonl --limit 300 "
     "--out outputs/run4-general/eval_report.json")
step("python evaluate.py --adapter outputs/run4 --test data/multilevel_holdout.jsonl "
     "--limit 18 --out outputs/run4-holdout/eval_report.json")

In [ ]:
# --- 7. Results, and save everything to the Output panel --------------------
import json, shutil
from pathlib import Path
out = Path("/kaggle/working")

for name, path in (("GENERAL (vs run 1's 84.0%)", "outputs/run4-general/eval_report.json"),
                   ("HAND-WRITTEN MULTI-LEVEL",   "outputs/run4-holdout/eval_report.json")):
    r = json.loads(Path(path).read_text())
    b, t = r["metrics"]["base"], r["metrics"]["tuned"]
    print(f"\n=== {name} — n={r['n_examples']} ===")
    print(f"  {'metric':<22}{'base':>10}{'run 4':>11}{'change':>10}")
    for k in b:
        print(f"  {k:<22}{b[k]:>9.1%}{t[k]:>10.1%}{(t[k]-b[k])*100:>+9.1f}")
    c = r.get("counts", {})
    print(f"  wins {c.get('tuned_win')} | regressions {c.get('tuned_regression')}")
    shutil.copy(path, out / (Path(path).parent.name + ".json"))

shutil.make_archive(str(out / "run4-adapter"), "zip", "outputs/run4")
print("\nSaved to /kaggle/working — download from the Output panel.")